# Baseline 00b: Long-context (stuff the whole corpus in)

Required deliverable (SPEC.md §7), not one of "the 10 patterns" -- no mandatory §8 template.
Instead of retrieving anything, this baseline concatenates the **entire pilot corpus** into the
model's context window and asks it to answer from that, using `gpt-5.4-mini-2026-03-17` (per
SPEC.md §19 Q3) for its large context window.

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`.** Outputs below prove the code
path runs end to end; they are not real long-context quality numbers.


## Reproducibility header (SPEC.md §11)

In [1]:
import platform
import sys
import subprocess
import openai
import numpy

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: 0a53e25169d7b633246d89b6ae127e8fb0ed03fd


## What this baseline does

All corpus chunks are joined into one context block and passed to every question -- no retrieval
step, so `retrieved_chunk_ids` is every chunk_id in the corpus for every question (the model saw
all of them, whether or not it used them). If the corpus is small enough to fit the model's
context window in full, this baseline shows what an "infinite recall" retrieval system would look
like -- useful as an upper bound for the retrieval-based patterns to compare against, and a real
alternative to RAG entirely for small corpora (see `docs/choosing.md`, written in P7).


In [2]:
import os
os.environ.setdefault("RAG_RECIPES_LLM", "mock")

import time
from pathlib import Path

from recipes import AnswerWithCitations
from recipes.llm import get_llm
from evals.run import load_corpus_by_id, load_qa_set, run_pattern

LONG_CONTEXT_MODEL = "gpt-5.4-mini-2026-03-17"
prompt_template = Path("../prompts/generation_prompt.txt").read_text(encoding="utf-8")


def make_long_context_pattern(corpus_by_id, llm):
    all_chunk_ids = list(corpus_by_id.keys())
    full_context = "\n\n".join(
        f"[{cid}] " + corpus_by_id[cid]["text"] for cid in all_chunk_ids
    )

    def retrieve_and_answer(question: str, k: int = 5) -> AnswerWithCitations:
        start = time.perf_counter()
        prompt = prompt_template.format(context=full_context, question=question)
        response = llm.complete(prompt=prompt, model=LONG_CONTEXT_MODEL, temperature=0.0)
        latency_ms = (time.perf_counter() - start) * 1000
        return AnswerWithCitations(
            answer=response.text,
            retrieved_chunk_ids=all_chunk_ids,
            latency_ms=latency_ms,
            input_tokens=response.input_tokens,
            output_tokens=response.output_tokens,
            cached_input_tokens=response.cached_input_tokens,
        )
    return retrieve_and_answer


## Run on our eval set

In [3]:
import os
from recipes.llm import MockLLM

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")

llm = get_llm()  # generation backend
if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm

pattern_fn = make_long_context_pattern(corpus_by_id, llm)

result = run_pattern(
    recipe_fn=pattern_fn,
    qa_set=qa_set,
    corpus_by_id=corpus_by_id,
    llm=judge_llm,
    pattern_name="00b_long_context_baseline",
    judges_enabled=True,
)


=== 00b_long_context_baseline (n=18) ===
  hit@3: 0.111  [95% CI 0.000, 0.278]
  hit@10: 0.278  [95% CI 0.111, 0.500]
  mrr: 0.183  [95% CI 0.062, 0.340]
  faithfulness: 1.000  [95% CI 1.000, 1.000]
  answer_relevance: 1.000  [95% CI 1.000, 1.000]
  citation_accuracy: 1.000  [95% CI 1.000, 1.000]
  filter_accuracy: 0.000  [95% CI 0.000, 0.000]
  p50_latency_ms: 0.1
  p95_latency_ms: 0.1
  usd_per_query: $0.03023
  eval_usd: $0.5442


## Discussion

**Correction (found during code review, see tasks/todo.md):** an earlier version of this section
claimed hit@k/mrr are "trivially 1.0 by construction" here, reasoning that every chunk is in
context so the relevant one is always "retrieved." That's wrong, and the executed cell above
disproves it directly: `hit_at_k()` slices `retrieved_chunk_ids[:k]` before checking for a match,
and `retrieved_chunk_ids` here is every corpus chunk_id in plain dict-insertion order -- NOT
ranked by relevance. Whether the truly-relevant chunk survives a top-k slice of an arbitrary
ordering is incidental, not guaranteed, which is exactly why the numbers above aren't 1.0.

hit@k/mrr still aren't a meaningful signal for this baseline -- they're an accident of corpus
ordering, not a measurement of anything this baseline is meant to test -- so this doesn't change
the practical takeaway: they measure retrieval, and there's no retrieval *step* to measure here.
What actually matters is
`faithfulness`, `answer_relevance`, `citation_accuracy`, `usd_per_query`, and latency: a real run
would show whether the model can actually *find* the right answer inside a large stuffed context
(long-context models are known to have "lost in the middle" failure modes), and at what cost/
latency compared to the retrieval-based patterns. On this ~54-chunk pilot corpus, the full corpus
almost certainly fits in `gpt-5.4-mini`'s context window without truncation; at the full 300-chunk
target from SPEC.md §4 this may no longer hold, and this notebook would need to report truncation
explicitly (per §19 Q3) rather than silently drop chunks.
